# Personality Atlas — Expert Review

**Validation Study for the Computational Atlas of Personality Models (TIST-2025-12-1243)**

This notebook provides the same evaluation tasks as the Streamlit app, but runs entirely in Google Colab — no Python installation required.

**Estimated time:** 45-60 minutes

**Recommended order:** Task 3 (10 min) → Task 2 (15 min) → Task 1 (25 min) → Download

Run each cell in order (Shift+Enter). Your responses are saved in memory and exported as CSVs at the end.

In [ ]:
#@title 1. Setup — Run this cell first
!pip install -q ipywidgets

import ipywidgets as widgets
from IPython.display import display, HTML
import pandas as pd
import json
import urllib.request
import io
import zipfile

BASE = "https://raw.githubusercontent.com/Wildertrek/survey/main/expert_evaluation/data/"

print("Loading evaluation data from GitHub...")
items_df = pd.read_csv(BASE + "item_classification_task.csv")
constructs_df = pd.read_csv(BASE + "construct_review_task.csv")
categories_df = pd.read_csv(BASE + "category_taxonomy_review.csv")
factor_guide = json.loads(urllib.request.urlopen(BASE + "factor_guide.json").read())
instructions_text = urllib.request.urlopen(BASE + "expert_evaluation_instructions.txt").read().decode()

# Clear all response columns for fresh evaluation
for col in ['expert_factor_assignment', 'expert_confidence', 'expert_notes']:
    items_df[col] = ''
for col in ['appropriateness_rating', 'notes']:
    constructs_df[col] = ''
for col in ['expert_appropriate', 'models_miscategorized', 'missing_models', 'notes']:
    categories_df[col] = ''

# Model card URLs for reference
ATLAS_BASE = "https://github.com/Wildertrek/survey/blob/main/atlas"
MODEL_CARD_MAP = {
    "OCEAN": "01_trait_based/ocean", "HEXACO": "01_trait_based/hexaco", "HEX": "01_trait_based/hexaco",
    "MBTI": "01_trait_based/mbti", "Four Temperaments": "01_trait_based/ftm",
    "NPI": "02_narcissism_based/npi", "Dark Triad": "02_narcissism_based/dtm",
    "Dark Tetrad": "02_narcissism_based/dt4", "HSNS": "02_narcissism_based/hsns",
    "NARQ": "02_narcissism_based/narq",
    "AAM": "03_motivational_value/aam", "SDT": "03_motivational_value/sdt",
    "RFT": "03_motivational_value/rft", "STBV": "03_motivational_value/stbv",
    "MST": "03_motivational_value/mst",
    "CEST": "04_cognitive_learning/cest",
    "MMPI": "05_clinical_health/mmpi", "SCID": "05_clinical_health/scid",
    "BDI": "05_clinical_health/bdi", "GAD-7": "05_clinical_health/gad7",
    "TCI": "05_clinical_health/tci",
    "TKI": "06_interpersonal_conflict/tki",
    "TEI": "07_application_holistic/tei", "Enneagram": "07_application_holistic/em",
    "RIASEC": "07_application_holistic/riasec",
}

def model_link(name):
    path = MODEL_CARD_MAP.get(name, '')
    if path:
        return f'<a href="{ATLAS_BASE}/{path}/MODEL_CARD.md" target="_blank">{name}</a>'
    return name

print(f"Loaded {len(items_df)} items, {len(constructs_df)} constructs, {len(categories_df)} categories")
print("Ready. Run the cells below to begin your evaluation.")

---
## Instructions

Run the cell below to view the full evaluation instructions.

In [ ]:
#@title 2. View Instructions
print(instructions_text)

---
## Task 3: Category Taxonomy Review (~10 min)

Review whether the 7-category structure is appropriate. For each category, indicate if the grouping is reasonable and note any issues.

Run the cell below, fill in your responses for each category, then click **Save All Task 3 Responses**.

In [ ]:
#@title 3. Task 3 — Taxonomy Review

task3_widgets = {}
all_t3 = []

for i, row in categories_df.iterrows():
    models_linked = ', '.join(model_link(m.strip()) for m in row['models'].split(','))
    info = widgets.HTML(value=(
        f'<div style="border:1px solid #555;padding:12px;margin:8px 0;border-radius:8px;background:#fff;color:#111">'
        f'<h3 style="margin-top:0;color:#111">{row["category"]} ({row["n_models"]} models)</h3>'
        f'<p><b>Models:</b> {models_linked}</p>'
        f'<p><b>Description:</b> {row["description"]}</p>'
        f'<details><summary><b>Theoretical Basis & Criteria (click to expand)</b></summary>'
        f'<p><b>Theoretical Basis:</b> {row.get("theoretical_basis", "")}</p>'
        f'<p><b>Inclusion Criteria:</b> {row.get("distinguishing_criteria", "")}</p>'
        f'</details>'
        f'<p style="background:#fff8e1;padding:8px;border-radius:4px;margin-top:8px;color:#111">'
        f'<b>Consider:</b> {row.get("key_considerations", "")}</p>'
        f'</div>'
    ))

    appropriate = widgets.RadioButtons(
        options=['Yes', 'Partially', 'No'],
        value=None,
        description='Reasonable?',
        style={'description_width': '100px'},
        layout=widgets.Layout(width='300px')
    )

    miscategorized = widgets.Text(
        description='Miscategorized:',
        placeholder='Models that belong in a different category',
        style={'description_width': '110px'},
        layout=widgets.Layout(width='95%')
    )

    missing = widgets.Text(
        description='Missing:',
        placeholder='Important models missing from this category',
        style={'description_width': '110px'},
        layout=widgets.Layout(width='95%')
    )

    notes = widgets.Textarea(
        description='Notes:',
        placeholder='Any observations about this category',
        style={'description_width': '110px'},
        layout=widgets.Layout(width='95%', height='60px')
    )

    task3_widgets[i] = {'appropriate': appropriate, 'miscategorized': miscategorized,
                        'missing': missing, 'notes': notes}

    all_t3.append(widgets.VBox([info, appropriate, miscategorized, missing, notes]))

t3_status = widgets.Output()

def save_task3(b):
    for i, ws in task3_widgets.items():
        categories_df.at[i, 'expert_appropriate'] = ws['appropriate'].value or ''
        categories_df.at[i, 'models_miscategorized'] = ws['miscategorized'].value
        categories_df.at[i, 'missing_models'] = ws['missing'].value
        categories_df.at[i, 'notes'] = ws['notes'].value
    done = sum(1 for _, r in categories_df.iterrows() if str(r.get('expert_appropriate', '')).strip())
    t3_status.clear_output()
    with t3_status:
        print(f'Task 3 saved! ({done}/7 categories reviewed)')

save_btn = widgets.Button(description='Save All Task 3 Responses', button_style='success',
                          layout=widgets.Layout(width='250px', height='40px'))
save_btn.on_click(save_task3)
all_t3.append(widgets.HBox([save_btn, t3_status]))

display(widgets.VBox(all_t3))

---
## Task 2: Construct Validity Review (~15 min)

Rate the appropriateness of the lexical schema (Adjective → Synonym → Verb → Noun) for each factor on a 1-5 scale.

Factors are grouped by model. Run the cell below, rate each factor, then click **Save All Task 2 Responses**.

In [ ]:
#@title 4. Task 2 — Construct Validity Review

task2_widgets = {}
all_t2 = []
current_model = None

for idx, row in constructs_df.iterrows():
    if row['model'] != current_model:
        current_model = row['model']
        all_t2.append(widgets.HTML(
            f'<h3 style="margin-top:20px;border-bottom:2px solid #1976d2;padding-bottom:4px;color:#1976d2">'
            f'{model_link(current_model)}</h3>'
        ))

    # Parse sample entries into HTML table
    entries = [e.strip() for e in str(row.get('sample_entries', '')).split('||') if e.strip()]
    table = '<table style="font-size:12px;border-collapse:collapse;margin:4px 0;color:#111;border:1px solid #ccc">'
    table += '<tr style="background:#e3f2fd;color:#111"><th style="padding:3px 8px">Adjective</th>'
    table += '<th style="padding:3px 8px">Synonym</th>'
    table += '<th style="padding:3px 8px">Verb</th>'
    table += '<th style="padding:3px 8px">Noun</th></tr>'
    for entry in entries:
        parts = [p.strip() for p in entry.split('/')]
        while len(parts) < 4:
            parts.append('')
        table += f'<tr style="color:#111;background:#fff"><td style="padding:2px 8px">{parts[0]}</td>'
        table += f'<td style="padding:2px 8px">{parts[1]}</td>'
        table += f'<td style="padding:2px 8px">{parts[2]}</td>'
        table += f'<td style="padding:2px 8px">{parts[3]}</td></tr>'
    table += '</table>'

    info = widgets.HTML(value=(
        f'<div style="padding:6px 0">'
        f'<b>{row["factor"]}</b> '
        f'<span style="color:#888">({len(entries)} sample entries from {row["n_schema_rows"]} rows)</span>'
        f'{table}</div>'
    ))

    rating = widgets.Dropdown(
        options=[('-- Select --', 0), ('1 - Inappropriate', 1), ('2 - Poor', 2),
                 ('3 - Adequate', 3), ('4 - Good', 4), ('5 - Excellent', 5)],
        value=0,
        description='Rating:',
        style={'description_width': '55px'},
        layout=widgets.Layout(width='220px')
    )

    notes = widgets.Text(
        description='Notes:',
        placeholder='Concerns or suggested replacements',
        style={'description_width': '50px'},
        layout=widgets.Layout(width='450px')
    )

    task2_widgets[idx] = {'rating': rating, 'notes': notes}
    all_t2.append(widgets.VBox([info, widgets.HBox([rating, notes])]))

t2_status = widgets.Output()

def save_task2(b):
    for idx, ws in task2_widgets.items():
        val = ws['rating'].value
        constructs_df.at[idx, 'appropriateness_rating'] = str(val) if val > 0 else ''
        constructs_df.at[idx, 'notes'] = ws['notes'].value
    done = sum(1 for _, r in constructs_df.iterrows()
              if str(r.get('appropriateness_rating', '')).strip()
              and str(r['appropriateness_rating']) not in ('', '0'))
    t2_status.clear_output()
    with t2_status:
        print(f'Task 2 saved! ({done}/{len(constructs_df)} factors rated)')

save_btn2 = widgets.Button(description='Save All Task 2 Responses', button_style='success',
                           layout=widgets.Layout(width='250px', height='40px'))
save_btn2.on_click(save_task2)
all_t2.append(widgets.HBox([save_btn2, t2_status]))

display(widgets.VBox(all_t2))

---
## Task 1: Item-Factor Classification (~25 min)

For each item, assign the most appropriate factor from the model's factor list.

- Some items are **reverse-scored** — classify by the construct being measured, not the direction of the statement.
- Some items are deliberately **ambiguous** — choose your best judgment and note alternatives.
- Items come from published instruments ("human") and LLM-generated sources.

Run the cell below. Use **Save & Next** to record your response and advance.

In [ ]:
#@title 5. Task 1 — Item Classification (one at a time)

task1_state = {'idx': 0, 'factor_widget': None}
task1_responses = {}

# Persistent widgets (never mutated)
item_area = widgets.Output()    # item info + factor dropdown rebuilt each time
progress_html = widgets.HTML()
status_out = widgets.Output()

confidence = widgets.IntSlider(
    min=1, max=5, value=3,
    description='Confidence:',
    style={'description_width': '80px'},
    layout=widgets.Layout(width='350px')
)
confidence_label = widgets.HTML('<span style="font-size:11px;color:#999">1=guessing, 5=certain</span>')

notes_input = widgets.Textarea(
    description='Notes:',
    placeholder='Optional — note alternative factors, concerns, etc.',
    style={'description_width': '70px'},
    layout=widgets.Layout(width='95%', height='60px')
)


def save_current():
    idx = task1_state['idx']
    fw = task1_state.get('factor_widget')
    if fw is None or fw.value is None:
        return
    task1_responses[idx] = {
        'factor': fw.value,
        'confidence': confidence.value,
        'notes': notes_input.value
    }
    items_df.at[idx, 'expert_factor_assignment'] = fw.value
    items_df.at[idx, 'expert_confidence'] = str(confidence.value)
    items_df.at[idx, 'expert_notes'] = notes_input.value


def show_item(idx):
    task1_state['idx'] = idx
    row = items_df.iloc[idx]
    n = len(items_df)

    done = len(task1_responses)
    pct = done / n * 100
    progress_html.value = (
        f'<div style="display:flex;align-items:center;gap:12px">'
        f'<b>Item {idx+1} of {n}</b>'
        f'<span style="color:#999">({done} answered, {pct:.0f}%)</span>'
        f'</div>'
    )

    # Build factor options with inline descriptors
    factors = [f.strip() for f in row['available_factors'].split('|')]
    guide = factor_guide.get(row['model'], {})
    labeled_options = [('-- Select factor --', None)]
    for f in factors:
        desc = guide.get(f, '')
        label = f'{f} \u2014 {desc}' if desc else f
        labeled_options.append((label, f))

    # Create a FRESH Dropdown each time (avoids ipywidgets mutation bugs)
    initial_value = None
    if idx in task1_responses:
        resp = task1_responses[idx]
        if resp['factor'] in [v for _, v in labeled_options]:
            initial_value = resp['factor']
        confidence.value = resp['confidence']
        notes_input.value = resp['notes']
    else:
        confidence.value = 3
        notes_input.value = ''

    fw = widgets.Dropdown(
        options=labeled_options,
        value=initial_value,
        description='Factor:',
        style={'description_width': '70px'},
        layout=widgets.Layout(width='95%')
    )
    task1_state['factor_widget'] = fw

    # Rebuild the item area
    instrument = row.get('instrument', '')
    inst_str = f' | <b>Instrument:</b> {instrument}' if pd.notna(instrument) and instrument else ''
    item_area.clear_output()
    with item_area:
        display(HTML(
            f'<div style="border:1px solid #555;padding:12px;border-radius:8px;background:#fff;color:#111">'
            f'<div style="display:flex;justify-content:space-between">'
            f'<span style="color:#555">{row["item_id"]}</span>'
            f'<span style="color:#555">{row.get("source_type", "unknown")}</span>'
            f'</div>'
            f'<p style="font-size:16px;background:#e3f2fd;padding:10px;border-radius:4px;margin:8px 0;color:#111">'
            f'\"{row["item_text"]}\"</p>'
            f'<p><b>Model:</b> {model_link(row["model"])}'
            f' | <b>Source:</b> {row.get("source_type", "unknown")}'
            f'{inst_str}'
            f' | <b>Factors:</b> {row["n_factors"]}</p>'
            f'</div>'
        ))
        display(fw)

    status_out.clear_output()


def on_prev(b):
    save_current()
    if task1_state['idx'] > 0:
        show_item(task1_state['idx'] - 1)

def on_next(b):
    save_current()
    if task1_state['idx'] < len(items_df) - 1:
        show_item(task1_state['idx'] + 1)

def on_save_next(b):
    fw = task1_state.get('factor_widget')
    if fw is None or fw.value is None:
        status_out.clear_output()
        with status_out:
            print('Please select a factor before saving.')
        return
    save_current()
    status_out.clear_output()
    with status_out:
        print(f'Saved {items_df.iloc[task1_state["idx"]]["item_id"]}')
    if task1_state['idx'] < len(items_df) - 1:
        show_item(task1_state['idx'] + 1)

def on_jump_unanswered(b):
    save_current()
    start = task1_state['idx']
    n = len(items_df)
    for offset in range(1, n):
        check = (start + offset) % n
        if check not in task1_responses:
            show_item(check)
            return
    status_out.clear_output()
    with status_out:
        print('All items answered!')

prev_btn = widgets.Button(description='Previous', layout=widgets.Layout(width='100px'))
next_btn = widgets.Button(description='Next', layout=widgets.Layout(width='100px'))
save_next_btn = widgets.Button(description='Save & Next', button_style='primary',
                               layout=widgets.Layout(width='140px', height='36px'))
jump_btn = widgets.Button(description='Jump to Unanswered', button_style='info',
                          layout=widgets.Layout(width='180px'))

prev_btn.on_click(on_prev)
next_btn.on_click(on_next)
save_next_btn.on_click(on_save_next)
jump_btn.on_click(on_jump_unanswered)

nav_bar = widgets.HBox([prev_btn, progress_html, next_btn, jump_btn])

form = widgets.VBox([
    nav_bar,
    item_area,
    widgets.HBox([confidence, confidence_label]),
    notes_input,
    widgets.HBox([save_next_btn]),
    status_out
])

show_item(0)
display(form)

---
## Download & Submit

Run the cell below to generate a zip file of your completed evaluation.

**Email the zip file to:** jraetano@utk.edu

In [ ]:
#@title 6. Download Results

# Make sure all Task 1 responses are saved
if 'task1_responses' in dir() and 'save_current' in dir():
    save_current()

# Summary
t1_done = sum(1 for _, r in items_df.iterrows() if str(r.get('expert_factor_assignment', '')).strip())
t2_done = sum(1 for _, r in constructs_df.iterrows()
             if str(r.get('appropriateness_rating', '')).strip()
             and str(r['appropriateness_rating']) not in ('', '0'))
t3_done = sum(1 for _, r in categories_df.iterrows() if str(r.get('expert_appropriate', '')).strip())

print('=' * 50)
print('EVALUATION SUMMARY')
print('=' * 50)
print(f'Task 1 (Item Classification): {t1_done}/{len(items_df)} items')
print(f'Task 2 (Construct Review):     {t2_done}/{len(constructs_df)} factors')
print(f'Task 3 (Taxonomy Review):      {t3_done}/{len(categories_df)} categories')
print()

total = len(items_df) + len(constructs_df) + len(categories_df)
done = t1_done + t2_done + t3_done
print(f'Overall: {done}/{total} ({done/total*100:.0f}%)')

if done < total:
    print(f'\nNote: {total - done} items incomplete. You can still submit a partial evaluation.')

# Generate zip
zip_buffer = io.BytesIO()
with zipfile.ZipFile(zip_buffer, 'w', zipfile.ZIP_DEFLATED) as zf:
    zf.writestr('item_classification_task.csv', items_df.to_csv(index=False))
    zf.writestr('construct_review_task.csv', constructs_df.to_csv(index=False))
    zf.writestr('category_taxonomy_review.csv', categories_df.to_csv(index=False))
zip_buffer.seek(0)

with open('expert_evaluation_results.zip', 'wb') as f:
    f.write(zip_buffer.getvalue())

print()
print('Zip file generated. Downloading...')

try:
    from google.colab import files as colab_files
    colab_files.download('expert_evaluation_results.zip')
    print('Download started!')
except ImportError:
    print('Not running in Colab. File saved as expert_evaluation_results.zip')

print()
print('Email the zip file to: jraetano@utk.edu')
print('Thank you for your time and expertise.')